# Notebook 11: Citation Dynamics by Sharing Granularity
Notebook 10 asks whether the yearly citations of sharing and non-sharing articles diverge over time. Here we ask the same question *within* the sharing articles: do the three granularities of shared data (*participant-*, *trial-* and *fixation-level*) differ in how their yearly citations change with the years since publication, `k`?<br>
The analysis is the one of notebook 10, run on the 80 sharing articles, with a `Sharing Class * k` interaction (two terms, PARTICIPANT is the baseline) in place of `Is Sharing * k`. The citation flow, the exclusion of the publication year and of the incomplete current year, the Gaussian on $log(1 + \text{Citations})$, the standardized time, the covariates and the lognormal back-transform of the figure are all as there; see notebook 10 for the reasoning.
<br><br>
**What differs, because there are only 80 articles:**
1. **No model search.** Notebook 10's winner for the shape of time, a linear term with a shared $log(k)$ term, is carried over. 80 articles are too few to choose a shape of time with, and the question here is whether the classes differ, not how time should be modelled.
2. **No venue random effect.** 9 of the 20 venues have a single sharing article, so the venue variance cannot be told from the residual (the reason notebooks 7 and 8 report their Model 1 for granularity); the venue's impact stays in as a covariate. The repeated measures within an article are still modelled, by a random effect of the article.
3. **A ladder of random-effects structures; the first usable rung is the primary model.** (1) A random intercept and a random slope of time per article, the maximal structure of notebook 10. (2) A random intercept per article only. (3) Pooled OLS with article-clustered standard errors: the "Model 1" of notebooks 7 and 8, made valid for repeated measures (plain OLS would treat the several years of one article as independent articles). A mixed rung is usable when it converged without a singular fit; the OLS always is. All three are reported side by side, so the conclusion can be checked against each.
4. **Tests.** The omnibus test of the class-by-time interaction (2 df) is what gets reported: a likelihood-ratio test for the mixed rungs and a Wald test for all three. The pairwise differences in slope and the class-specific slopes are shown with their intervals for transparency, and are not read one by one when the omnibus is null.

The classes have 23 to 31 articles each and very few observations after the first years, so a null result here means that the data cannot tell the classes apart, not that the classes are the same: read the widths of the intervals, not only the p-values.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import plotly.graph_objects as go
import plotly.io as pio

from helpers.config import *
from helpers import dataset
from helpers.dynamics import (
    ALPHA, PRIMARY_FAMILY, PRIMARY_OUTCOME, ModelSpec, add_class_dummies, add_time_terms, build_dynamics_frame, class_slopes,
    fit_clustered_ols, fit_with_fallback, group_by_time, is_usable, lrt_omnibus, omnibus_table, prediction_grid, to_mean_citations,
)
from helpers.plotting import DynamicsGroup, plot_citation_dynamics, save_figure
from helpers.r_models import fit_glmm
from helpers.stats import contrasts_from_estimates

pio.renderers.default = "browser"

## Setup

In [ ]:
BASELINE = "PARTICIPANT"
CANDIDATE = "k + log(k)"        # notebook 10's winner for g(k), carried over and not re-selected: 80 articles are too few to select with

SPEC = ModelSpec(
    group="(is_trial + is_fixation)",       # two 0/1 dummies, PARTICIPANT is the baseline
    candidates={CANDIDATE: dict(time="{group} * k_z + log_k_z", no_interaction="{group} + k_z + log_k_z", slope="k_z")},
    covariates="has_us_author + is_open_access + has_preprint + log_number_of_authors + venue_impact + log_weeks_since_pub",
    random_effects={      # the venue is not a random effect here: 9 of the 20 venues have a single sharing article
        "full": "(1 + {slope} | article)",
        "intercept": "(1 | article)",
    },
)

### Load Data

In [ ]:
combined, FEATURES_DF = dataset.load_or_build()

## Sharing Granularity
### Prepare Data
One row per article-year of the 80 sharing articles. The time terms are standardized with notebook 10's scale (all 234 articles), so that a coefficient per SD of `k` means the same in both notebooks, and `sharing_class` is coded as two 0/1 dummies with PARTICIPANT as the baseline. An article that shared at several levels is classified at the finest one (fixation, then trial, then participant), as everywhere in this project.<br>
The classes are unevenly spread over time (below): the last years have a handful of articles per class, and TRIAL has none at the last year.

In [ ]:
DYNAMICS_DF = build_dynamics_frame(combined, FEATURES_DF)
_, TIME_SCALE = add_time_terms(DYNAMICS_DF)       # notebook 10's standardization (all 234 articles), so that "per SD of k" means the same here

SHARERS = DYNAMICS_DF[DYNAMICS_DF["is_sharing_data"] == 1]
MODEL_DATA, DUMMIES = add_class_dummies(add_time_terms(SHARERS, TIME_SCALE)[0], baseline=BASELINE)
DUMMIES        # dummy -> class, in the order of the model's terms

In [ ]:
articles = MODEL_DATA.drop_duplicates("article")
print(f"{len(articles)} articles, {len(MODEL_DATA)} article-years ({MODEL_DATA.groupby('article').size().min()} to {MODEL_DATA.groupby('article').size().max()} years per article)")
print(f"{articles['venue_name'].nunique()} venues, {int((articles['venue_name'].value_counts() == 1).sum())} of them with a single sharing article")

# articles observed in each year after publication, per class; the tail is thin
class_sizes = MODEL_DATA.groupby(["sharing_class", "k"]).size()
class_sizes.unstack("k").reindex([BASELINE, *DUMMIES.values()]).fillna(0).astype(int)

In [ ]:
# observed mean yearly citations per class and year, with the standard error of the mean
observed_by_year = MODEL_DATA.groupby(["sharing_class", "k"])["citations"].agg(["mean", "sem"])
observed_by_year["mean"].unstack("k").reindex([BASELINE, *DUMMIES.values()]).round(2)

### Main Analysis
$$ \text{log}(1 + \text{Citations}_{i,k}) \sim C(\text{Sharing Class}) \times k + log(k) + \text{Has US Author} + \text{Is OA} + \text{Has Preprint} + log(\text{Number of Authors}) + \text{Impact Score} + log(\text{Weeks Since Pub.}) + (1 + k | \text{Article}) $$
The predictions are for a typical article: the covariates at their mean over the sharing articles and the random effects at zero, for every year in which at least one sharing article is observed.

In [ ]:
# typical-article predictions (covariates at their mean over the sharers, random effects at zero), per class and year
CLASSES = pd.DataFrame({"sharing_class": [BASELINE, *DUMMIES.values()], **{dummy: [int(level == name) for level in [BASELINE, *DUMMIES.values()]] for dummy, name in DUMMIES.items()}})
PREDICTION_GRID = prediction_grid(MODEL_DATA, TIME_SCALE, groups=CLASSES, covariates=SPEC.covariate_names)
PREDICTION_GRID.head(3)

#### Fit Models
Each rung is fitted with the class-by-time interaction. The two mixed rungs are also fitted without it, for the likelihood-ratio test (all by maximum likelihood, so that the likelihoods can be compared).

In [ ]:
fits = {
    "full": fit_glmm(MODEL_DATA, SPEC.formula(PRIMARY_OUTCOME, CANDIDATE, "full"), PRIMARY_FAMILY, newdata=PREDICTION_GRID),
    "intercept": fit_glmm(MODEL_DATA, SPEC.formula(PRIMARY_OUTCOME, CANDIDATE, "intercept"), PRIMARY_FAMILY, newdata=PREDICTION_GRID),
    "OLS": fit_clustered_ols(SPEC, MODEL_DATA, PRIMARY_OUTCOME, CANDIDATE, newdata=PREDICTION_GRID),
}
# the same models without the interaction, for the likelihood-ratio test (the OLS has no likelihood test: its omnibus is a Wald test)
null_fits = {
    name: fit_glmm(MODEL_DATA, SPEC.formula(PRIMARY_OUTCOME, CANDIDATE, name, interaction=False), PRIMARY_FAMILY)
    for name in ("full", "intercept")
}

#### Which Structure, and What Does the Interaction Test Say?
The primary model is the first usable rung of the ladder. The table gives, for each rung, whether it is usable (`converged`, `singular`), its omnibus test of the class-by-time interaction, and its R² and AIC. The conclusion should not depend on the rung: if it does, that is the finding. The pooled OLS has no random effects, so its much worse AIC is the cost of ignoring that an article is observed several times (and its R² is the plain OLS R²).

In [ ]:
# the primary model is the first usable rung of the ladder (the pooled OLS always is)
primary_structure = next(name for name in fits if is_usable(fits[name]))
primary = fits[primary_structure]

structure_table = omnibus_table({name: (fit, name) for name, fit in fits.items()})
structure_table["AIC"] = [fit.fit["aic"] for fit in fits.values()]
for name, null_fit in null_fits.items():
    test = lrt_omnibus(fits[name], null_fit, df=len(DUMMIES))
    structure_table.loc[name, ["LRT chi2", "LRT p"]] = [test["statistic"], test["p"]]
status = {name: "usable" if is_usable(fit) else "NOT usable (singular or not converged)" for name, fit in fits.items()}
print(f"Primary model: {primary_structure}; the rungs are {status}")
structure_table

In [ ]:
random_effects = fits["full"].variance_components
random_effects

#### Differences in Time Trend Between the Classes
The class-by-time coefficients are each class's difference in slope from PARTICIPANT, so the difference between two non-baseline classes needs the covariance of the estimates. The three pairwise differences (Benjamini-Hochberg across the three, as in notebooks 7 and 8) and each class's own slope of the linear time term follow. The slopes are in the model's units (per SD of `k`) and per year; the log-time term is shared by all classes, so these are the slopes of the part of time that can differ between classes.

In [ ]:
# the class-by-time interaction terms are the differences in slope from the baseline, so the contrasts between classes come from their covariance
interactions = {f"{dummy}:k_z": level for dummy, level in DUMMIES.items()}
df = primary.fit.get("df_denom")        # the t reference of the pooled OLS (clusters - 1); the mixed model's contrasts use the normal
slope_contrasts = contrasts_from_estimates(primary.fixed_effects["estimate"], primary.fixed_effects_cov, interactions, BASELINE, df=df)
slope_contrasts

In [ ]:
class_slope_table = class_slopes(primary, DUMMIES, BASELINE, "k_z", TIME_SCALE.k_sd)
class_slope_table

#### Model Output

In [ ]:
z = stats.norm.ppf(0.975)
fixed_effects = primary.fixed_effects.copy()
fixed_effects["ci_low"] = fixed_effects["estimate"] - z * fixed_effects["se"]
fixed_effects["ci_high"] = fixed_effects["estimate"] + z * fixed_effects["se"]
fixed_effects

In [ ]:
primary.variance_components

## Robustness
The three rungs above are the first robustness check: they are three ways of handling the repeated measures, and the omnibus conclusion is read against all of them. Below, the primary rung is refitted with different offsets in $log(y + c)$ and with the zeros dropped (which also changes the estimand: it conditions on the article being cited that year), without the last, thin, years (`k >= 8`), and as a Poisson count model. Each refit falls back down the ladder if its first rung is not usable, and the table shows the structure it used.<br>
We do not repeat notebook 10's check of the age adjustment (which does not move the result there) or the negative binomial (which did not converge with random slopes there). The omnibus is a Wald test in every row, since a likelihood-ratio test would need a second fit per row.

In [ ]:
LADDER = ("intercept",) if primary_structure == "intercept" else ("full", "intercept")      # each refit may fall back too


def refit(data: pd.DataFrame, outcome: str, family: str = PRIMARY_FAMILY):
    """The primary rung's refit of a robustness check: the pooled OLS if that is the primary, otherwise the ladder."""
    if primary_structure == "OLS" and family == PRIMARY_FAMILY:
        return fit_clustered_ols(SPEC, data, outcome, CANDIDATE), "OLS"
    return fit_with_fallback(SPEC, data, outcome, CANDIDATE, family, structures=LADDER)


outcome_fits = {"log(y + 1) (primary)": (primary, primary_structure)}
for offset in (0.5, 0.1):
    outcome_fits[f"log(y + {offset})"] = refit(MODEL_DATA.assign(outcome=np.log(MODEL_DATA["citations"] + offset)), "outcome")
positive = MODEL_DATA[MODEL_DATA["citations"] > 0]
outcome_fits["log(y), zeros dropped"] = refit(positive.assign(outcome=np.log(positive["citations"])), "outcome")
outcome_fits["without k >= 8 (thin tail)"] = refit(MODEL_DATA[MODEL_DATA["k"] < 8], PRIMARY_OUTCOME)
omnibus_table(outcome_fits)

In [ ]:
count_fits = {"poisson": refit(MODEL_DATA, "citations", "poisson")}
omnibus_table(count_fits)

## Visualize Model Predictions
The same figure as notebook 10, with three groups: the model's predictions for a typical article (a line with filled markers), the raw observed means (open markers), and a table of the article counts. The classes are told apart by color and marker, the predictions from the observed means by the fill. The observed means are unadjusted, while the curves are adjusted for the covariates, so a gap between them is expected where the classes differ in venue, preprints and the like.<br>
The predictions are shown only for the years in which the class has articles (there is no TRIAL article at the last year, so no curve is drawn there), and the table shows how thin the late years are: a point backed by two or three articles is mostly the model's smooth shape. The table sits in the bottom-left corner, where the curves and the observed means leave room, and the y axis starts at zero.

In [ ]:
SHOW_ERROR_BANDS = False    # the 95% confidence band around each curve
SHOW_ERROR_BARS = True      # +/- 1 SEM (the standard error of the model's prediction) at each point
SHOW_SAMPLE_SIZES = True    # an inset table with the number of articles behind each point, per class and year
SHOW_OBSERVED_MEANS = True  # the raw (unadjusted) mean yearly citations +/- SEM, as open markers beside each point

# back-transform from log(1 + y) to the mean on the citation scale (the lognormal correction, see `to_mean_citations`)
predictions = to_mean_citations(primary.predictions, primary.variance_components, SPEC.candidates[CANDIDATE]["slope"])
predictions = predictions.merge(class_sizes.rename("articles").reset_index(), on=["sharing_class", "k"])      # only the years in which the class has articles

# the class is told apart by color and marker, and the observed means (open) from the predictions (filled) by the fill
groups = [
    DynamicsGroup(
        level.title(), CLASS_COLORS[level], marker, predictions[predictions["sharing_class"] == level],
        observed_by_year.loc[level], class_sizes.loc[level],
    )
    for level, marker in zip([BASELINE, *DUMMIES.values()], ("circle", "square", "diamond"))
]
dynamics_fig = plot_citation_dynamics(
    groups, title="Citation Dynamics by Sharing Granularity", show_error_bands=SHOW_ERROR_BANDS, show_error_bars=SHOW_ERROR_BARS,
    show_observed_means=SHOW_OBSERVED_MEANS, show_counts=SHOW_SAMPLE_SIZES, height=560, table_corner="left",
)

if False:
    dynamics_fig.show()

In [ ]:
# flip to True to export this figure into `output/`
if False:
    save_figure(dynamics_fig, "citation_dynamics_granularity.png", width=800, height=560)

## Limits
- **Low power.** With 23 to 31 articles per class, the interval of a slope difference is wide: only a large difference could be detected, so a null omnibus is weak evidence of equality. Report the intervals.
- **A thin tail.** After the first four years each class has a dozen articles or fewer, and the last years a handful; the curves there rest on the shape of the time term. The `k >= 8` refit checks that the conclusion does not hinge on them.
- **Class and cohort are not independent.** TRIAL is concentrated in the 2021 cohort, so its later years come from few, young articles. The age term (log weeks since publication) adjusts for this only partly.
- **Not a model search.** The shape of time was chosen in notebook 10 on all articles and reused here; the class-by-time interaction is linear in time, next to a log-time term shared by the classes.
- **Observational.** The classes are the granularities authors chose to share at, not an assignment.